In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# 1 Handling Missing Data

The way that missing data is represented in pandas objects is somewhat imperfect, but it is sufficient for most real-world use. For data with `float64` dtype, pandas uses the floating-point value `NaN` (Not a Number) to represent missing data.  

We call this a sentinel value: when present, it indicates a missing (or null) value:

In [ ]:
float_data = pd.Series([1.2, -3.5, np.nan, 0])

float_data

,0
0,1.2
1,-3.5
2,NaN
3,0.0


The `isna` method gives us a Boolean Series with `True` where values are null:

In [ ]:
float_data.isna()

,0
0,False
1,False
2,True
3,False


The built-in Python `None` value is also treated as NA:

In [ ]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data.isna()

,0
0,False
1,True
2,True
3,False


## 1.1 Filtering Out Missing Data

There are a few ways to filter out missing data. While you always have the option to do it by hand using `pandas.isna` and Boolean indexing, `dropna` can be helpful. On a Series, it returns the Series with only the nonnull data and index values:

In [ ]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

,0
0,1.0
2,3.5
4,7.0


This is the same thing as doing:

In [ ]:
data[data.notna()]

,0
0,1.0
2,3.5
4,7.0


With DataFrame objects, there are different ways to remove missing data. You may want to drop rows or columns that are all NA, or only those rows or columns containing any NAs at all. `dropna` by default drops any row containing a missing value:

In [ ]:
data = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan], [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [ ]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


Passing `how="all"` will drop only rows that are all NA:

In [ ]:
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


Keep in mind that these functions return new objects by default and do not modify the contents of the original object.

To drop columns in the same way, pass `axis="columns"`:

In [ ]:
data[4] = np.nan

data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [ ]:
data.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


Suppose you want to keep only rows containing at most a certain number of missing observations. You can indicate this with the `thresh` argument:

In [ ]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))

df.iloc[:4, 1] = np.nan
df.iloc[:2, 2] = np.nan

df

,0,1,2
0,-0.812023,NaN,NaN
1,0.002966,NaN,NaN
2,0.723360,NaN,0.184606
3,0.235094,NaN,-0.384495
4,1.470860,0.971330,1.258171
5,1.150873,-1.168976,-0.492149
6,0.810437,-0.567418,-2.656016


In [ ]:
df.dropna()

,0,1,2
4,1.470860,0.971330,1.258171
5,1.150873,-1.168976,-0.492149
6,0.810437,-0.567418,-2.656016


In [ ]:
df.dropna(thresh=2)

,0,1,2
2,0.723360,NaN,0.184606
3,0.235094,NaN,-0.384495
4,1.470860,0.971330,1.258171
5,1.150873,-1.168976,-0.492149
6,0.810437,-0.567418,-2.656016


## 1.2 Filling In Missing Data

Calling `fillna` with a constant replaces missing values with that value:

In [ ]:
df.fillna(0)

,0,1,2
0,-0.812023,0.000000,0.000000
1,0.002966,0.000000,0.000000
2,0.723360,0.000000,0.184606
3,0.235094,0.000000,-0.384495
4,1.470860,0.971330,1.258171
5,1.150873,-1.168976,-0.492149
6,0.810437,-0.567418,-2.656016


Calling `fillna` with a dictionary, you can use a different fill value for each column:

In [ ]:
df.fillna({1: 0.5, 2: 2})

,0,1,2
0,-0.812023,0.500000,2.000000
1,0.002966,0.500000,2.000000
2,0.723360,0.500000,0.184606
3,0.235094,0.500000,-0.384495
4,1.470860,0.971330,1.258171
5,1.150873,-1.168976,-0.492149
6,0.810437,-0.567418,-2.656016


The same interpolation methosd available for reindexing can be used with `fillna`

In [ ]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))

df.iloc[2:, 1] = np.nan

df.iloc[4:, 2] = np.nan

df

,0,1,2
0,-0.581261,-0.133104,0.010335
1,2.221589,0.796780,0.065444
2,-0.513185,NaN,-1.066284
3,2.427205,NaN,1.034240
4,1.640751,NaN,NaN
5,0.605668,NaN,NaN


In [ ]:
df.fillna(method="ffill")

/tmp/ipython-input-3944122520.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill")


,0,1,2
0,-0.581261,-0.133104,0.010335
1,2.221589,0.796780,0.065444
2,-0.513185,0.796780,-1.066284
3,2.427205,0.796780,1.034240
4,1.640751,0.796780,1.034240
5,0.605668,0.796780,1.034240


In [ ]:
df.fillna(method="ffill", limit=2)

/tmp/ipython-input-1627181726.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", limit=2)


,0,1,2
0,-0.581261,-0.133104,0.010335
1,2.221589,0.796780,0.065444
2,-0.513185,0.796780,-1.066284
3,2.427205,0.796780,1.034240
4,1.640751,NaN,1.034240
5,0.605668,NaN,1.034240


With `fillna` you can do lots of other things such as simple data imputation using the median or mean statistics:

In [ ]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])

data.fillna(data.mean())

,0
0,1.000000
1,3.833333
2,3.500000
3,3.833333
4,7.000000


# 2 Data Transformation

## 2.1 Removing Duplicates

Duplicate rows may be found in a DataFrame for any number of reasons. Here is an example:

In [ ]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                   "k2": [1, 1, 2, 3, 3, 4, 4]})

data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


The DataFrame method `duplicated` returns a Boolean Series indicating whether each row is a duplicate (its column values are exactly equal to those in an earlier row) or not:

In [ ]:
data.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
5,False
6,True


Relatedly, `drop_duplicates` returns a DataFrame with rows where the duplicated array is `False` filtered out:

In [ ]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both methods by default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates. Suppose we had an additional column of values and wanted to filter duplicates based only on the `"k1"` column:

In [ ]:
data["v1"] = range(7)

data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [ ]:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


`duplicated` and `drop_duplicates` by default keep the first observed value combination. Passing `keep="last"` will return the last one:

In [ ]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## 2.2 Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values in an array, Series, or column in a DataFrame. Consider the following hypothetical data collected about various kinds of meat:

In [ ]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})

data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suppose you wanted to add a column indicating the type of animal that each food came from. Let’s write down a mapping of each distinct meat type to the kind of animal:

In [ ]:
meat_to_animal = {
    "bacon": "pig",
    "pulled pork": "pig",
    "pastrami": "cow",
    "corned beef": "cow",
    "honey ham": "pig",
    "nova lox": "salmon"
}

The `map` method on a Series accepts a function or dictionary-like object containing a mapping to do the transformation of values:

In [ ]:
data["animal"] = data["food"].map(meat_to_animal)

data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


We could also have passed a function that does all the work:

In [ ]:
def get_animal(x):
  return meat_to_animal[x]

data["food"].map(get_animal)

,food
0,pig
1,pig
2,pig
3,cow
4,cow
5,pig
6,cow
7,pig
8,salmon


> Using `map` is a convenient way to perform element-wise transformations and other data cleaning-related operations.



## 2.3 Replacing Values

Filling in missing data with the `fillna` method is a special case of more general value replacement. As you've already seen, `map` can be used to modify a subset of values in an object, but `replace` provides a simpler and more flexible way to do so. Let’s consider this Series:

In [ ]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])

data

,0
0,1.0
1,-999.0
2,2.0
3,-999.0
4,-1000.0
5,3.0


The `-999` values might be sentinel values for missing data. To replace these with NA values that pandas understands, we can use `replace`, producing a new Series:

In [ ]:
data.replace(-999, np.nan)

,0
0,1.0
1,NaN
2,2.0
3,NaN
4,-1000.0
5,3.0


If you want to replace multiple values at once, you instead pass a list and then the substitute value:

In [ ]:
data.replace([-999, -1000], np.nan)

,0
0,1.0
1,NaN
2,2.0
3,NaN
4,NaN
5,3.0


To use a different replacement for each value, pass a list of substitutes:

In [ ]:
data.replace([-999, -1000], [np.nan, 0])

,0
0,1.0
1,NaN
2,2.0
3,NaN
4,0.0
5,3.0


The argument passed can also be a dictionary:

In [ ]:
data.replace({-999: np.nan, -1000: 0})

,0
0,1.0
1,NaN
2,2.0
3,NaN
4,0.0
5,3.0


## 2.4 Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects. You can also modify the axes in place without creating a new data structure.

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])

def transform(x):
  return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

You can assign to the `index` attribute, modifying the DataFrame in place:

In [ ]:
data.index = data.index.map(transform)

data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


If you want to create a transformed version of a dataset without modifying the original, a useful method is `rename`:

In [ ]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notably, `rename` can be used in conjunction with a dictionary-like object, providing new values for a subset of the axis labels:

In [ ]:
data.rename(index={"OHIO": "INDIANA"},
            columns={"three": "peekaboo"})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


> `rename` saves you from the chore of copying the DataFrame manually and assigning new values to its `index` and `columns` attributes.

## 2.5 Discretization and Binning

Continuous data is often discretized or otherwise separated into “bins” for analysis. Suppose you have data about a group of people in a study, and you want to group them into discrete age buckets:

In [ ]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Let’s divide these into bins of 18 to 25, 26 to 35, 36 to 60, and finally 61 and older. To do so, you have to use `pandas.cut`:

In [ ]:
bins = [18, 25, 35, 60, 100]

age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In the string representation of an interval, a parenthesis means that the side is open (exclusive), while the square bracket means it is closed (inclusive). You can change which side is closed by passing `right=False`:

In [ ]:
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

You can override the default interval-based bin labeling by passing a list or array to the `labels` option:

In [ ]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

If you pass an integer number of bins to `pandas.cut` instead of explicit bin edges, it will compute equal-length bins based on the minimum and maximum values in the data. Consider the case of some uniformly distributed data chopped into fourths:

In [ ]:
data = np.random.uniform(size=20)

pd.cut(data, 4, precision = 2)

[(0.28, 0.52], (0.76, 1.0], (0.76, 1.0], (0.52, 0.76], (0.047, 0.28], ..., (0.28, 0.52], (0.28, 0.52], (0.76, 1.0], (0.28, 0.52], (0.047, 0.28]]
Length: 20
Categories (4, interval[float64, right]): [(0.047, 0.28] < (0.28, 0.52] < (0.52, 0.76] < (0.76, 1.0]]

> The `precision=2` option limits the decimal precision to two digits.

A closely related function, `pandas.qcut`, bins the data based on sample quantiles. Depending on the distribution of the data, using pandas.cut will not usually result in each bin having the same number of data points. Since `pandas.qcut` uses sample quantiles instead, you will obtain roughly equally sized bins:

In [ ]:
data = np.random.standard_normal(1000)

quartiles = pd.qcut(data, 4, precision=2)

quartiles

[(0.7, 3.23], (-0.64, 0.064], (-0.64, 0.064], (0.7, 3.23], (-2.9299999999999997, -0.64], ..., (-2.9299999999999997, -0.64], (-0.64, 0.064], (-2.9299999999999997, -0.64], (-0.64, 0.064], (0.7, 3.23]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.9299999999999997, -0.64] < (-0.64, 0.064] < (0.064, 0.7] <
                                           (0.7, 3.23]]

In [ ]:
pd.value_counts(quartiles)

/tmp/ipython-input-3472704981.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(quartiles)


,count
"(-2.9299999999999997, -0.64]",250
"(-0.64, 0.064]",250
"(0.064, 0.7]",250
"(0.7, 3.23]",250


Similar to `pandas.cut`, you can pass your own quantiles (numbers between 0 and 1, inclusive):

In [ ]:
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.]).value_counts()

,count
"(-2.925, -1.299]",100
"(-1.299, 0.0635]",400
"(0.0635, 1.313]",400
"(1.313, 3.231]",100


## 2.6 Detecting and Filtering Outliers

Filtering or transforming outliers is largely a matter of applying array operations. Consider a DataFrame with some normally distributed data:

In [ ]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.042000,-0.045478,0.012169,-0.022236
std,1.011141,1.021694,1.001959,1.021717
min,-3.217063,-3.295183,-3.141089,-3.215439
25%,-0.609829,-0.756742,-0.701662,-0.669379
50%,0.039420,-0.058011,0.021319,-0.052054
75%,0.757450,0.617425,0.700857,0.666600
max,2.841501,2.979594,2.989624,4.132889


Suppose you wanted to find values in one of the columns exceeding 3 in absolute value:

In [ ]:
col = data[2]

col[col.abs() > 3]

,2
758,-3.141089
883,-3.021094


To select all rows having a value exceeding 3 or –3, you can use the `any` method on a Boolean DataFrame:

In [ ]:
data[(data.abs() > 3).any(axis="columns")]

,0,1,2,3
141,-3.217063,-0.358984,-1.363174,-0.301273
565,0.366977,-3.031300,-0.251368,1.704470
643,-0.877742,0.725307,-0.237233,4.132889
739,0.296270,0.848029,1.076269,-3.215439
758,0.211838,0.722837,-3.141089,-0.455377
883,-1.660267,0.241153,-3.021094,-1.205141
947,0.095007,-3.295183,1.654553,-0.157947
950,-3.146145,0.188800,0.570181,0.434270
985,-1.345902,-3.109981,-0.433653,0.837622


Values can be set based on these criteria. Here is code to cap values outside the interval –3 to 3:

In [ ]:
data[data.abs() > 3] = np.sign(data) * 3

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.042364,-0.045042,0.012331,-0.023154
std,1.010012,1.020381,1.001461,1.017071
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.609829,-0.756742,-0.701662,-0.669379
50%,0.039420,-0.058011,0.021319,-0.052054
75%,0.757450,0.617425,0.700857,0.666600
max,2.841501,2.979594,2.989624,3.000000


The statement `np.sign(data)` produces 1 and –1 values based on whether the values in `data` are positive or negative:

In [ ]:
np.sign(data).head()

,0,1,2,3
0,-1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,-1.0
2,-1.0,1.0,1.0,1.0
3,1.0,-1.0,-1.0,-1.0
4,-1.0,1.0,1.0,-1.0


## 2.7 Permutation and Random Sampling

Permuting (randomly reordering) a Series or the rows in a DataFrame is possible using the `numpy.random.permutation` function. Calling `permutation` with the length of the axis you want to permute produces an array of integers indicating the new ordering:

In [ ]:
df = pd.DataFrame(np.arange(5*7).reshape((5, 7)))

df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [ ]:
sampler = np.random.permutation(5)

sampler

array([2, 1, 0, 3, 4])

That array can then be used in `iloc`-based indexing or the equivalent `take` function:

In [ ]:
df.take(sampler)

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
0,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [ ]:
df.iloc[sampler]

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
0,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


By invoking take with `axis="columns"`, we could also select a permutation of the columns:

In [ ]:
column_sampler = np.random.permutation(7)

column_sampler

array([6, 2, 1, 0, 4, 3, 5])

In [ ]:
df.take(column_sampler, axis="columns")

,6,2,1,0,4,3,5
0,6,2,1,0,4,3,5
1,13,9,8,7,11,10,12
2,20,16,15,14,18,17,19
3,27,23,22,21,25,24,26
4,34,30,29,28,32,31,33


To select a random subset without replacement (the same row cannot appear twice), you can use the `sample` method on Series and DataFrame:

In [ ]:
df.sample(n=3)

,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27


To generate a sample with replacement (to allow repeat choices), pass `replace=True` to sample:

In [ ]:
choices = pd.Series([5, 7, -1, 6 ,4])

choices.sample(n=10, replace=True)

,0
0,5
0,5
3,6
3,6
2,-1
1,7
2,-1
0,5
4,4
2,-1


## 2.8 Computing Indicator/Dummy Variables

Another type of transformation for statistical modeling or machine learning applications is converting a categorical variable into a dummy or indicator matrix. If a column in a DataFrame has `k` distinct values, you would derive a matrix or DataFrame with `k` columns containing all 1s and 0s. pandas has a `pandas.get_dummies` function for doing this, though you could also devise one yourself. Let’s consider an example DataFrame:

In [ ]:
df = pd.DataFrame({
    "key": ["b", "b", "a", "c", "a", "b"],
    "data1": range(6)
})

df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [ ]:
pd.get_dummies(df["key"], dtype=float)

,a,b,c
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
5,0.0,1.0,0.0


Here I passed `dtype=float` to change the output type from boolean (the default in more recent versions of pandas) to floating point.

In some cases, you may want to add a prefix to the columns in the indicator DataFrame, which can then be merged with the other data. `pandas.get_dummies` has a prefix argument for doing this:

In [ ]:
dummies = pd.get_dummies(df["key"], prefix="key", dtype=float)

df_with_dummy = df[["data1"]].join(dummies)

df_with_dummy

,data1,key_a,key_b,key_c
0,0,0.0,1.0,0.0
1,1,0.0,1.0,0.0
2,2,1.0,0.0,0.0
3,3,0.0,0.0,1.0
4,4,1.0,0.0,0.0
5,5,0.0,1.0,0.0


If a row in a DataFrame belongs to multiple categories, we have to use a different approach to create the dummy variables. Let’s look at the MovieLens 1M dataset, which is investigated in more detail in Ch 13: Data Analysis Examples:

In [ ]:
mnames = ["movie_id", "title", "genres"]

movies = pd.read_table("/content/drive/MyDrive/Colab Notebooks/Python for Data Analysis/datasets/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")

movies[:10]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Python for Data Analysis/datasets/movielens/movies.dat'

`str.get_dummies()` 是pandas中一个非常实用的字符串处理方法，用于将包含多个分类标签的字符串列转换为虚拟变量（dummy variables）矩阵。

In [ ]:
dummies = movies["genres"].str.get_dummies("|")

dummies.iloc[:10, :6]

Then, as before, you can combine this with movies while adding a `"Genre_"` to the column names in the `dummies` DataFrame with the `add_prefix` method:

In [ ]:
movies_windic = movies.join(dummies.add_prefix("Genre_"))

movies_windic.iloc[0]

A useful recipe for statistical applications is to combine `pandas.get_dummies` with a discretization function like `pandas.cut`:

In [ ]:
np.random.seed(12345) # to make the example repeatable

values = np.random.uniform(size=10)

values

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins))

# 3 String Manipulation

## 3.1 Python Built-In String Object Methods

In many string munging and scripting applications, built-in string methods are sufficient. As an example, a comma-separated string can be broken into pieces with `split`:

In [ ]:
val = "a,b,  guido"

val.split(",")

['a', 'b', '  guido']

`split` is often combined with `strip` to trim whitespace (including line breaks):

In [ ]:
pieces = [x.strip() for x in val.split(",")]

pieces

['a', 'b', 'guido']

These substrings could be concatenated together with a two-colon delimiter using addition:

In [ ]:
first, second, third = pieces

first + "::" + second + "::" + third

'a::b::guido'

But this isn’t a practical generic method. A faster and more Pythonic way is to pass a list or tuple to the `join` method on the string `"::"`:

In [ ]:
"::".join(pieces)

'a::b::guido'

Other methods are concerned with locating substrings. Using Python’s `in` keyword is the best way to detect a substring, though `index` and `find` can also be used:

In [ ]:
"guido" in val

True

In [ ]:
val.index(",")

1

In [ ]:
val.find(":")

-1

Note that the difference between `find` and `index` is that `index` raises an exception if the string isn’t found (versus returning –1):

In [ ]:
val.index(":")

ValueError: substring not found

Relatedly, `count` returns the number of occurrences of a particular substring:

In [ ]:
val.count(",")

2

`replace` will substitute occurrences of one pattern for another. It is commonly used to delete patterns, too, by passing an empty string:

In [ ]:
val.replace(",", "::")

'a::b::  guido'

In [ ]:
val.replace(",", "")

'ab  guido'

## 3.2 Regular Expressions

The `re` module functions fall into three categories: pattern matching, substitution, and splitting. Naturally these are all related; a regex describes a pattern to locate in the text, which can then be used for many purposes. Let’s look at a simple example: suppose we wanted to split a string with a variable number of whitespace characters (tabs, spaces, and newlines).

The regex describing one or more whitespace characters is `\s+`:

In [ ]:
import re

text = "foo     bar\t baz  \tqux"

re.split(r"\s+", text)

['foo', 'bar', 'baz', 'qux']

When you call `re.split(r"\s+", text)`, the regular expression is first compiled, and then its `split` method is called on the passed text. You can compile the regex yourself with `re.compile`, forming a reusable regex object

In [ ]:
regex = re.compile(r"\s+")

regex.split(text)

['foo', 'bar', 'baz', 'qux']

If, instead, you wanted to get a list of all patterns matching the regex, you can use the `findall` method:

In [ ]:
regex.findall(text)

['     ', '\t ', '  \t']

`match` and `search` are closely related to `findall`. While findall returns all matches in a string, `search` returns only the first match. More rigidly, `match` only matches at the beginning of the string. As a less trivial example, let’s consider a block of text and a regular expression capable of identifying most email addresses:

In [ ]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""

pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"

# re.IGNORECASE makes the regex case insensitive
regex = re.compile(pattern, flags=re.IGNORECASE)

Using `findall` on the text produces a list of the email addresses:

In [ ]:
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

`search` returns a special match object for the first email address in the text. For the preceding regex, the match object can only tell us the start and end position of the pattern in the string:

In [ ]:
m = regex.search(text)

m

<re.Match object; span=(5, 20), match='dave@google.com'>

In [ ]:
text[m.start():m.end()]

'dave@google.com'

`regex.match` returns None, as it will match only if the pattern occurs at the start of the string:

In [ ]:
print(regex.match(text))

None


Relatedly, `sub` will return a new string with occurrences of the pattern replaced by a new string:

`regex.sub("替换内容", 原始文本)` 会将匹配到的所有模式替换为指定的新字符串。

In [ ]:
print(regex.sub("REDACTED", text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


Suppose you wanted to find email addresses and simultaneously segment each address into its three components: username, domain name, and domain suffix. To do this, put parentheses around the parts of the pattern to segment:

In [ ]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

regex = re.compile(pattern, flags=re.IGNORECASE)

A match object produced by this modified regex returns a tuple of the pattern components with its `groups` method:

In [ ]:
m = regex.match("wesm@bright.net")

m.groups()

('wesm', 'bright', 'net')

`findall` returns a list of tuples when the pattern has groups:

In [ ]:
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

`sub` also has access to groups in each match using special symbols like `\1` and `\2`. The symbol `\1` corresponds to the first matched group, `\2` corresponds to the second, and so forth:

In [ ]:
print(regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com


## 3.3 String Functions in pandas

Cleaning up a messy dataset for analysis often requires a lot of string manipulation. To complicate matters, a column containing strings will sometimes have missing data:

In [ ]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
         "Rob": "rob@gmail.com", "Wes": np.nan}

data = pd.Series(data)

data

,0
Dave,dave@google.com
Steve,steve@gmail.com
Rob,rob@gmail.com
Wes,NaN


In [ ]:
data.isna()

,0
Dave,False
Steve,False
Rob,False
Wes,True


we could check whether each email address has `"gmail"` in it with `str.contains`:

In [ ]:
data.str.contains("gmail")

,0
Dave,False
Steve,True
Rob,True
Wes,NaN


Note that the result of this operation has an object dtype. pandas has extension types that provide for specialized treatment of strings, integers, and Boolean data which until recently have had some rough edges when working with missing data:

In [ ]:
data_as_string_ext = data.astype('string')

data_as_string_ext

,0
Dave,dave@google.com
Steve,steve@gmail.com
Rob,rob@gmail.com
Wes,<NA>


In [ ]:
data_as_string_ext.str.contains("gmail")

,0
Dave,False
Steve,True
Rob,True
Wes,<NA>


Regular expressions can be used, too, along with any `re` options like `IGNORECASE`:

In [ ]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

data.str.findall(pattern, flags=re.IGNORECASE)

,0
Dave,"[(dave, google, com)]"
Steve,"[(steve, gmail, com)]"
Rob,"[(rob, gmail, com)]"
Wes,NaN


There are a couple of ways to do vectorized element retrieval. Either use `str.get` or index into the `str` attribute:


In [ ]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]

matches

,0
Dave,"(dave, google, com)"
Steve,"(steve, gmail, com)"
Rob,"(rob, gmail, com)"
Wes,NaN


In [ ]:
matches.str.get(1)

,0
Dave,google
Steve,gmail
Rob,gmail
Wes,NaN


You can similarly slice strings using this syntax:

In [ ]:
data.str[:5]

,0
Dave,dave@
Steve,steve
Rob,rob@g
Wes,NaN


The `str.extract` method will return the captured groups of a regular expression as a DataFrame:

In [ ]:
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


# 4 Categorical Data

## 4.1 Background and Motivation

Frequently, a column in a table may contain repeated instances of a smaller set of distinct values. We have already seen functions like unique and value_counts, which enable us to extract the distinct values from an array and compute their frequencies, respectively:

In [ ]:
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)

values

,0
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


In [ ]:
pd.unique(values)

array(['apple', 'orange'], dtype=object)

In [ ]:
pd.value_counts(values)

/tmp/ipython-input-3297668723.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(values)


,count
apple,6
orange,2


In [ ]:
values = pd.Series([0, 1, 0, 0] * 2)

dim = pd.Series(['apple', 'orange'])

values

,0
0,0
1,1
2,0
3,0
4,0
5,1
6,0
7,0


In [ ]:
dim

,0
0,apple
1,orange


We can use the `take` method to restore the original Series of strings:

In [ ]:
dim.take(values)

,0
0,apple
1,orange
0,apple
0,apple
0,apple
1,orange
0,apple
0,apple


## 4.2 Categorical Extension Type in pandas

pandas has a special `Categorical` extension type for holding data that uses the integer-based categorical representation or encoding. This is a popular data compression technique for data with many occurrences of similar values and can provide significantly faster performance with lower memory use, especially for string data.

In [ ]:
fruits = ['apple', 'orange', 'apple', 'apple'] * 2

N = len(fruits)

rng = np.random.default_rng(seed=12345)

df = pd.DataFrame({'fruit': fruits,
                   'basket_id': np.arange(N),
                   'count': rng.integers(3, 15, size=N),
                   'weight': rng.uniform(0, 4, size=N)},
                  columns = ['basket_id', 'fruit', 'count', 'weight'])

df

,basket_id,fruit,count,weight
0,0,apple,11,1.564438
1,1,orange,5,1.331256
2,2,apple,12,2.393235
3,3,apple,6,0.746937
4,4,apple,5,2.691024
5,5,orange,12,3.767211
6,6,apple,10,0.992983
7,7,apple,11,3.795525


Here, `df['fruit']` is an array of Python string objects. We can convert it to categorical by calling:

In [ ]:
fruit_cat = df['fruit'].astype('category')

fruit_cat

,fruit
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


The values for `fruit_cat` are now an instance of `pandas.Categorical`, which you can access via the `.array` attribute:

In [ ]:
c = fruit_cat.array

type(c)

pandas.core.arrays.categorical.Categorical

In [ ]:
c.categories

Index(['apple', 'orange'], dtype='object')

In [ ]:
c.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

A useful trick to get a mapping between codes and categories is:

In [ ]:
dict(enumerate(c.categories))

{0: 'apple', 1: 'orange'}

You can convert a DataFrame column to categorical by assigning the converted result:

In [ ]:
df['fruit'] = df['fruit'].astype('category')

df['fruit']

,fruit
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


You can also create `pandas.Categorical` directly from other types of Python sequences:

In [ ]:
my_categories = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])

my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

If you have obtained categorical encoded data from another source, you can use the alternative `from_codes` constructor:

In [ ]:
categories = ['foo', 'bar', 'baz']

codes = [0, 1, 2, 0, 0, 1]

my_cats_2 = pd.Categorical.from_codes(codes, categories)

my_cats_2

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

Unless explicitly specified, categorical conversions assume no specific ordering of the categories. So the `categories` array may be in a different order depending on the ordering of the input data. When using `from_codes` or any of the other constructors, you can indicate that the categories have a meaningful ordering:

In [ ]:
ordered_cat = pd.Categorical.from_codes(codes, categories, ordered=True)

ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

The output `[foo < bar < baz]` indicates that 'foo' precedes 'bar' in the ordering, and so on. An unordered categorical instance can be made ordered with as_ordered:

In [ ]:
my_cats_2.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

## 4.3 Computations with Categoricals

In [ ]:
rng = np.random.default_rng(seed=12345)

draws = rng.standard_normal(1000)

draws[:5]

array([-1.42382504,  1.26372846, -0.87066174, -0.25917323, -0.07534331])

Let's compute a quartile binning of this data and extract some statistics:

In [ ]:
bins = pd.qcut(draws, 4)

bins

[(-3.121, -0.675], (0.687, 3.211], (-3.121, -0.675], (-0.675, 0.0134], (-0.675, 0.0134], ..., (0.0134, 0.687], (0.0134, 0.687], (-0.675, 0.0134], (0.0134, 0.687], (-0.675, 0.0134]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.121, -0.675] < (-0.675, 0.0134] < (0.0134, 0.687] <
                                           (0.687, 3.211]]

While useful, the exact sample quartiles may be less useful for producing a report than quartile names. We can achieve this with the `labels` argument to `qcut`

In [ ]:
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

bins

['Q1', 'Q4', 'Q1', 'Q2', 'Q2', ..., 'Q3', 'Q3', 'Q2', 'Q3', 'Q2']
Length: 1000
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [ ]:
bins.codes[:10]

array([0, 3, 0, 1, 1, 0, 0, 2, 2, 0], dtype=int8)

The labeled `bins` categorical does not contain information about the bin edges in the data, so we can use `groupby` to extract some summary statistics:

In [ ]:
bins = pd.Series(bins, name='quartile')

results = (pd.Series(draws)
           .groupby(bins)
           .agg(['count', 'min', 'max'])
           .reset_index())

results

/tmp/ipython-input-1053954477.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(bins)


,quartile,count,min,max
0,Q1,250,-3.119609,-0.678494
1,Q2,250,-0.673305,0.008009
2,Q3,250,0.018753,0.686183
3,Q4,250,0.688282,3.211418


## 4.4 Better performance with categoricals

At the beginning of the section, I said that categorical types can improve performance and memory use, so let's look at some examples. Consider some Series with 10 million elements and a small number of distinct categories:

In [ ]:
N = 10_000_000

labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))

# Now we convert labels to categrical
categories = labels.astype('category')

Now we note that labels uses significantly more memory than categories:

In [ ]:
labels.memory_usage(deep=True)

520000132

In [ ]:
categories.memory_usage(deep=True)

10000512

## 4.5 Categorical Methods

Series containing categorical data have several special methods similar to the Series.str specialized string methods. This also provides convenient access to the categories and codes. Consider the Series:

In [ ]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)

cat_s = s.astype('category')

cat_s

,0
0,a
1,b
2,c
3,d
4,a
5,b
6,c
7,d


The special accessor attribute `cat` provides access to categorical methods:

In [ ]:
cat_s.cat.codes

,0
0,0
1,1
2,2
3,3
4,0
5,1
6,2
7,3


In [ ]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

Suppose that we know the actual set of categories for this data extends beyond the four values observed in the data. We can use the `set_categories` method to change them:

In [ ]:
actual_categories = ['a', 'b', 'c', 'd', 'e']

cat_s2 = cat_s.cat.set_categories(actual_categories)

cat_s2

,0
0,a
1,b
2,c
3,d
4,a
5,b
6,c
7,d


While it appears that the data is unchanged, the new categories will be reflected in operations that use them. For example, `value_counts` respects the categories, if present:

In [ ]:
cat_s.value_counts()

,count
a,2
b,2
c,2
d,2


In [ ]:
cat_s2.value_counts()

,count
a,2
b,2
c,2
d,2
e,0


In large datasets, categoricals are often used as a convenient tool for memory savings and better performance. After you filter a large DataFrame or Series, many of the categories may not appear in the data. To help with this, we can use the `remove_unused_categories` method to trim unobserved categories:

In [ ]:
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]

cat_s3

,0
0,a
1,b
4,a
5,b


In [ ]:
cat_s3.cat.remove_unused_categories()

,0
0,a
1,b
4,a
5,b


## 4.6 Creating dummy variables for modeling

When you're using statistics or machine learning tools, you'll often transform categorical data into dummy variables, also known as one-hot encoding. This involves creating a DataFrame with a column for each distinct category; these columns contain 1s for occurrences of a given category and 0 otherwise.

In [ ]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')

As mentioned previously in this chapter, the `pandas.get_dummies` function converts this one-dimensional categorical data into a DataFrame containing the dummy variable:

In [ ]:
pd.get_dummies(cat_s, dtype=float)

,a,b,c,d
0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,1.0
